In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import confusion_matrix,precision_score,recall_score,f1_score,accuracy_score
from sklearn.preprocessing import  OneHotEncoder,StandardScaler,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import seaborn as sns
from sklearn.tree import  DecisionTreeClassifier
from sklearn.pipeline import  Pipeline

Churn =Yes(1)--> Customer left the telecom service
Churn = Yes(0)--> Customer continued telecom service

In [2]:
df=pd.read_csv("telco_data.csv")

In [3]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            6293 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           6043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            4543 non-null   float64
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   6043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       5543 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [5]:
df.isnull().sum()

customerID             0
gender               750
SeniorCitizen          0
Partner             1000
Dependents             0
tenure              2500
PhoneService           0
MultipleLines          0
InternetService     1000
OnlineSecurity         0
OnlineBackup           0
DeviceProtection       0
TechSupport            0
StreamingTV         1500
StreamingMovies        0
Contract               0
PaperlessBilling       0
PaymentMethod          0
MonthlyCharges      1500
TotalCharges           0
Churn                  0
dtype: int64

In [6]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1.0,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34.0,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2.0,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45.0,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2.0,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [7]:
def clean_charges(val):
    val=val.strip()
    #return None if val=='' else float(val)
    if val=='':
        return None
    return float(val)

In [8]:
df['TotalCharges']=df['TotalCharges'].apply(clean_charges)

In [9]:
df=df.drop(columns=['customerID'])

In [10]:
df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1.0,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34.0,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2.0,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45.0,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2.0,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24.0,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,No
7039,Female,0,Yes,Yes,72.0,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No
7040,Female,0,Yes,Yes,11.0,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,Male,1,Yes,No,NaN,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes


In [11]:
x=df.drop('Churn',axis=1)
y=df.Churn


In [12]:
num_cols=x.select_dtypes(include="number").columns
obj_cols=x.select_dtypes(exclude="number").columns

In [13]:
num_cols


Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')

In [14]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42)

In [15]:
# df['TotalCharges']=df['TotalCharges'].str.strip()
# df['TotalCharges']=df['TotalCharges'].replace('',None)
# df['TotalCharges']=df['TotalCharges'].astype(float)

### Simple Imputer
It replace the missing values with the help of strategy('mean,median,most_frequent') or constant with fill_value.
It accepts only 2D

In [16]:
# imputer = SimpleImputer(strategy='most_frequent')
# imputer = SimpleImputer(strategy='constant',fill_value='unknown')

In [17]:
# df[['gender']]=imputer.fit_transform(df[['gender']])

In [18]:
# df[df['gender'].isna()]

In [19]:
# df['gender'].value_counts()

Grid Search cv with pipeline

In [20]:
numeric_preprocessing=Pipeline(
    steps=[
        ('simple_imputer',SimpleImputer(strategy='mean')),
        ('standard_scaler',StandardScaler())
    ]
)
numeric_preprocessing.fit_transform(xtrain[num_cols])

array([[-0.4377492 ,  0.        ,  0.        , -0.42210502],
       [-0.4377492 ,  0.        ,  0.        ,  1.25536015],
       [-0.4377492 ,  0.        ,  0.        , -1.00299144],
       ...,
       [-0.4377492 , -0.93381312, -1.46464784, -0.87799925],
       [ 2.28441306, -0.93381312,  1.15806793, -0.48254445],
       [-0.4377492 , -0.29753207, -1.50986708, -0.81110232]],
      shape=(5634, 4))

In [21]:
object_preprocessing=Pipeline(
    steps=[
        ('simple_imputer',SimpleImputer(strategy='constant',fill_value='unknown')),
        ('ordinalEncoder',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1))]
)
object_preprocessing.fit_transform(xtrain[obj_cols])

array([[0., 0., 1., ..., 1., 0., 3.],
       [0., 0., 0., ..., 2., 1., 0.],
       [1., 1., 0., ..., 0., 1., 2.],
       ...,
       [1., 1., 1., ..., 0., 1., 2.],
       [1., 0., 0., ..., 0., 1., 2.],
       [1., 0., 0., ..., 1., 0., 1.]], shape=(5634, 15))

In [22]:
preprocessing = ColumnTransformer(
    transformers=[
        ("numeric_preprocessing",numeric_preprocessing,num_cols),
        ("objectpreprocessing",object_preprocessing,obj_cols)

    ]
)
preprocessing.fit_transform(xtrain)

array([[-0.4377492 ,  0.        ,  0.        , ...,  1.        ,
         0.        ,  3.        ],
       [-0.4377492 ,  0.        ,  0.        , ...,  2.        ,
         1.        ,  0.        ],
       [-0.4377492 ,  0.        ,  0.        , ...,  0.        ,
         1.        ,  2.        ],
       ...,
       [-0.4377492 , -0.93381312, -1.46464784, ...,  0.        ,
         1.        ,  2.        ],
       [ 2.28441306, -0.93381312,  1.15806793, ...,  0.        ,
         1.        ,  2.        ],
       [-0.4377492 , -0.29753207, -1.50986708, ...,  1.        ,
         0.        ,  1.        ]], shape=(5634, 19))

In [23]:
num_cols

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')

In [24]:
pipeline=Pipeline(   
    steps=[
        ("preprocessing",preprocessing),
        ('model',LogisticRegression())
    ]
)
# Pipeline.fit_transform(obj_cols,ytrain)
pipeline.fit(xtrain,ytrain)
# pipeline.named_steps['model']
# pipeline.transform(num_cols)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric_preprocessing', ...), ('objectpreprocessing', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [25]:
decisiontree_pipeline=Pipeline(
    steps=[
('preprocessing',preprocessing),
('model',DecisionTreeClassifier())
    ]
)

In [26]:
decisiontree_pipeline.fit(xtrain,ytrain)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric_preprocessing', ...), ('objectpreprocessing', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [27]:
decisiontree_pipeline.score(xtrain,ytrain)

0.998757543485978

In [28]:
pipeline.score(xtrain,ytrain)

0.7953496627618033

In [29]:
grid_search_cv=GridSearchCV(
    estimator=pipeline,
    param_grid={'model__C':[0.01,0.1,1.0,10],
    'model__penalty':['l1','l2'],
    'model__solver':['liblinear'],
    'model__class_weight':['balanced']
    },

    cv=10,
    n_jobs=-1,
    verbose=1,
    scoring='f1_macro')
    

In [30]:
grid_search_cv.fit(xtrain,ytrain)

Fitting 10 folds for each of 8 candidates, totalling 80 fits


,estimator,Pipeline(step...egression())])
,param_grid,"{'model__C': [0.01, 0.1, ...], 'model__class_weight': ['balanced'], 'model__penalty': ['l1', 'l2'], 'model__solver': ['liblinear']}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,10
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('numeric_preprocessing', ...), ('objectpreprocessing', ...)]"


In [31]:
grid_search_cv.best_params_

{'model__C': 0.1,
 'model__class_weight': 'balanced',
 'model__penalty': 'l2',
 'model__solver': 'liblinear'}

In [32]:
cv_result=pd.DataFrame(grid_search_cv.cv_results_)

In [33]:
cv_result.sort_values(by='rank_test_score').reset_index(drop=True)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__C,param_model__class_weight,param_model__penalty,param_model__solver,params,split0_test_score,...,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score
0,0.202838,0.066478,0.051318,0.022053,0.10,balanced,l2,liblinear,"{'model__C': 0.1, 'model__class_weight': 'bala...",0.732850,...,0.715996,0.704887,0.710897,0.716613,0.683812,0.722244,0.698934,0.708371,0.013606,1
1,0.158785,0.031457,0.041092,0.012568,1.00,balanced,l2,liblinear,"{'model__C': 1.0, 'model__class_weight': 'bala...",0.728922,...,0.713524,0.710407,0.706067,0.714320,0.686139,0.717366,0.700532,0.707860,0.013013,2
2,0.245204,0.026916,0.038597,0.010944,1.00,balanced,l1,liblinear,"{'model__C': 1.0, 'model__class_weight': 'bala...",0.727291,...,0.713524,0.712704,0.706067,0.714320,0.686139,0.715746,0.700532,0.707607,0.012991,3
3,0.154737,0.011909,0.041074,0.010772,10.00,balanced,l2,liblinear,"{'model__C': 10, 'model__class_weight': 'balan...",0.728922,...,0.713524,0.712704,0.706067,0.714320,0.687720,0.715746,0.698934,0.707607,0.013004,4
4,0.237291,0.033849,0.038697,0.016837,10.00,balanced,l1,liblinear,"{'model__C': 10, 'model__class_weight': 'balan...",0.728922,...,0.711911,0.712704,0.706067,0.714320,0.687720,0.715746,0.698934,0.707446,0.012940,5
5,0.195161,0.048015,0.033716,0.008448,0.10,balanced,l1,liblinear,"{'model__C': 0.1, 'model__class_weight': 'bala...",0.730555,...,0.707938,0.708102,0.706820,0.711090,0.684627,0.727019,0.691766,0.705908,0.014562,6
6,0.165504,0.031253,0.037193,0.011392,0.01,balanced,l2,liblinear,"{'model__C': 0.01, 'model__class_weight': 'bal...",0.728510,...,0.711996,0.705790,0.713267,0.714994,0.682860,0.732067,0.689373,0.705705,0.016550,7
7,0.123640,0.020548,0.040026,0.010907,0.01,balanced,l1,liblinear,"{'model__C': 0.01, 'model__class_weight': 'bal...",0.721296,...,0.673920,0.704887,0.714884,0.695037,0.678325,0.723753,0.694113,0.698198,0.018385,8


In [34]:
model=grid_search_cv.best_estimator_

In [35]:
model.fit(xtrain,ytrain)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric_preprocessing', ...), ('objectpreprocessing', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [36]:
grid_search_cv.score(xtrain,ytrain)

0.711780298486697

In [37]:
grid_search_cv.predict(xtest)

array(['Yes', 'No', 'No', ..., 'No', 'No', 'Yes'],
      shape=(1409,), dtype=object)

In [38]:
grid_search_cv.best_score_

np.float64(0.708371134830436)

In [39]:
model.score(xtrain,ytrain)

0.7410365637202698